# 05a — Decoding pilot (choose anti-repetition settings before full generation)

Accelerator: **GPU T4 x2**, Internet **on**. Expected time: ~15–20 min.
**Inputs:** attach the saved SFT notebook (02) and DPO notebook (03) outputs (*Add Input → Your Work*).

Greedy decoding makes these small models loop. This pilot generates from SFT, DPO-UF and DPO-HH on
`eval_sets/decoding_pilot_20.json` — 20 prompts that are **not** in any evaluation set — under four
settings, and scores repetition automatically. The chosen setting then goes into `configs/generation.yaml`
and is used identically for every model in the final comparison.

In [ ]:
REPO_URL = "https://github.com/madgod2004/small-model-alignment-starter.git"
REPO_DIR = "/kaggle/working/small-model-alignment-starter"
BRANCH = "main"

In [ ]:
import os
if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors matplotlib "peft>=0.11.0" "trl>=0.12.0" pytest
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
!python scripts/check_env.py --expect_gpus 2

## 1. Locate checkpoints

In [ ]:
import glob, json, os

def find_model_dir(run, kind="checkpoints/best"):
    """Locate outputs/<run>/<kind>/model.safetensors inside the attached Kaggle inputs."""
    hits = sorted(glob.glob(f"/kaggle/input/**/outputs/{run}/{kind}/model.safetensors", recursive=True))
    return os.path.dirname(hits[0]) if hits else None

MODELS = {name: find_model_dir(name) for name in ["sft", "dpo_hh", "dpo_uf"]}
for name, path in MODELS.items():
    print(f"{name:8} -> {path}")
assert all(MODELS.values()), "attach the SFT (02) and DPO (03) notebook outputs as inputs"

## 2. Generate: 3 models x 4 decoding settings (jobs spread over both GPUs)

In [ ]:
SETTINGS = {
    "greedy": {},
    "rp1.1": {"repetition_penalty": 1.1},
    "rp1.2": {"repetition_penalty": 1.2},
    "ngram4": {"no_repeat_ngram_size": 4},
}
jobs = []
for model, path in MODELS.items():
    for setting, extra in SETTINGS.items():
        jobs.append({"model_path": path, "generator": f"{model}__{setting}", "eval_set": "eval_sets/decoding_pilot_20.json",
                     "output_path": f"outputs/pilot/{model}__{setting}.json", **extra})
os.makedirs("outputs/pilot", exist_ok=True)
json.dump(jobs, open("outputs/pilot/jobs.json", "w"), indent=1)
!PYTHONUNBUFFERED=1 python scripts/run_generation_jobs.py --jobs outputs/pilot/jobs.json

## 3. Repetition statistics
`trunc` = share hitting max_new_tokens, `rep_line` = mean share of repeated lines, `any_rep` = share of
responses with any repeated line, `dist3` = distinct word-trigram ratio (higher = less repetition).

In [ ]:
!python scripts/generation_stats.py outputs/pilot/*__*.json --json_out outputs/pilot/summary.json

## 4. Read a few prompts across settings (check that penalties do not damage lists, code, or facts)

In [ ]:
outputs = {s: json.load(open(f"outputs/pilot/sft__{s}.json")) for s in SETTINGS}
for i in [0, 3, 7, 9, 12, 15]:
    print("=" * 100)
    print(outputs["greedy"][i]["instruction"][:200])
    for s in SETTINGS:
        print(f"--- {s}:", outputs[s][i]["output"][:450])

## 5. Archive

In [ ]:
!zip -qr /kaggle/working/decoding_pilot_outputs.zip outputs/pilot outputs/logs/gen_*__*.log
!ls -lh /kaggle/working/*.zip